# Week 11–12: Dimensionality Reduction — PCA & t-SNE

[![PCA](https://img.shields.io/badge/PCA-Principal%20Component%20Analysis-blue.svg)](https://scikit-learn.org/stable/modules/decomposition.html#pca)
[![t-SNE](https://img.shields.io/badge/t--SNE-Visualization-green.svg)](https://scikit-learn.org/stable/modules/manifold.html#t-sne)
[![scikit-learn](https://img.shields.io/badge/scikit--learn-1.x-orange.svg)](https://scikit-learn.org/)

---

## 📌 Syllabus Overview (SIT-N Hack-o-Week 5th Semester)

| Topic | Description |
|-------|-------------|
| **PCA** | Principal Component Analysis — linear dimensionality reduction, explained variance, feature compression |
| **t-SNE** | t-distributed Stochastic Neighbor Embedding — non-linear visualization, local neighborhood preservation |

### Learning Objectives
1. Understand the intuition behind PCA and how it reduces dimensionality by projecting onto principal components
2. Implement PCA, analyze explained variance, and visualize high-dimensional data in 2D
3. Understand t-SNE intuition — how it preserves local structure for visualization
4. Compare PCA (global/linear) vs t-SNE (local/non-linear) on the same dataset
5. Explore the effect of hyperparameters (n_components, perplexity) on results

---

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries loaded successfully!')

---

## 2. Load & Explore the Dataset

We use the **Kaggle Student Performance** dataset (same as Week 9–10). The numeric academic features are ideal for demonstrating dimensionality reduction.

In [ ]:
import os

# Try multiple possible paths for the dataset
possible_paths = [
    os.path.join('..', 'WEEK9 - WEEK10', 'data', 'kaggle_student_performance.csv'),
    os.path.join('..', '..', 'WEEK9 - WEEK10', 'data', 'kaggle_student_performance.csv'),
    os.path.join('..', 'data', 'kaggle_student_performance.csv'),
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'✅ Dataset loaded from: {path}')
        break

if df is None:
    raise FileNotFoundError('Could not find kaggle_student_performance.csv')

print(f'Shape: {df.shape[0]} students × {df.shape[1]} features')
df.head()

In [ ]:
# Select numeric features for dimensionality reduction
FEATURES = ['CGPA', 'AttendanceRate', 'StudyHoursPerWeek', 'MathScore', 'ReadingScore', 'WritingScore']

X = df[FEATURES].apply(pd.to_numeric, errors='coerce')
if X.isna().any().any():
    print('⚠️ Missing values found, filling with median...')
    X = X.fillna(X.median())

# Create a target label for coloring (not used in fitting)
df['Distinction'] = (df['CGPA'] >= 8.5).astype(int)

print(f'\n📊 Feature matrix shape: {X.shape}')
print(f'   Features: {FEATURES}')
print(f'   Distinction distribution: {dict(df["Distinction"].value_counts())}')
X.describe().round(2)

### 2.1 Correlation Heatmap

Before applying PCA, let's check the correlation structure. PCA is most effective when features are correlated — it can compress correlated features into fewer components.

In [ ]:
plt.figure(figsize=(8, 6))
corr_matrix = X.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Highly correlated features can be compressed by PCA into fewer principal components.')

---

## 3. Data Standardization

**Why standardize?** PCA finds directions of maximum variance. If features have different scales (e.g., CGPA 0–10 vs MathScore 0–100), features with larger ranges would dominate the principal components.

StandardScaler transforms each feature to have **mean = 0** and **standard deviation = 1**.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Before scaling:')
print(f'  Mean: {X.mean().values.round(2)}')
print(f'  Std:  {X.std().values.round(2)}')
print(f'\nAfter scaling:')
print(f'  Mean: {X_scaled.mean(axis=0).round(6)}')
print(f'  Std:  {X_scaled.std(axis=0).round(6)}')
print('\n✅ All features now have mean ≈ 0 and std ≈ 1')

---

## 4. Principal Component Analysis (PCA)

### 4.1 PCA Intuition

PCA is a **linear** dimensionality reduction technique that:

1. **Finds new axes** (principal components) that are linear combinations of the original features
2. **PC1** captures the direction of maximum variance in the data
3. **PC2** captures the maximum remaining variance, orthogonal to PC1
4. Each subsequent component captures decreasing amounts of variance
5. The **explained variance ratio** tells us how much information each component retains

**Key properties:**
- Principal components are **uncorrelated** (orthogonal)
- PCA preserves **global structure** (variance patterns)
- It's useful for feature reduction, noise removal, and visualization

### 4.2 Fit PCA with All Components

In [ ]:
# Fit PCA with all components to analyze explained variance
pca_full = PCA()
pca_full.fit(X_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print('═' * 55)
print('   PCA EXPLAINED VARIANCE ANALYSIS')
print('═' * 55)
print(f'{"Component":<12} {"Individual":>12} {"Cumulative":>12}')
print('-' * 38)
for i, (ind, cum) in enumerate(zip(explained_variance, cumulative_variance)):
    marker = ' ◀ 95% threshold' if i > 0 and cumulative_variance[i-1] < 0.95 and cum >= 0.95 else ''
    print(f'  PC{i+1:<8} {ind:>11.2%} {cum:>11.2%}{marker}')
print('═' * 55)

### 4.3 Explained Variance Plot (Scree Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

components = np.arange(1, len(explained_variance) + 1)

# Left: Individual explained variance (Scree plot)
axes[0].bar(components, explained_variance, color='#6366f1', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot — Individual Variance', fontweight='bold')
axes[0].set_xticks(components)
for i, v in enumerate(explained_variance):
    axes[0].text(i + 1, v + 0.005, f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')

# Right: Cumulative explained variance
axes[1].plot(components, cumulative_variance, 'o-', color='#10b981', linewidth=2, markersize=8)
axes[1].fill_between(components, cumulative_variance, alpha=0.15, color='#10b981')
axes[1].axhline(y=0.95, color='#f43f5e', linestyle='--', linewidth=1.5, label='95% threshold')
axes[1].axhline(y=0.90, color='#f59e0b', linestyle='--', linewidth=1.5, label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].set_xticks(components)
axes[1].set_ylim(0, 1.05)
axes[1].legend(loc='lower right')
for i, v in enumerate(cumulative_variance):
    axes[1].text(i + 1, v + 0.02, f'{v:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Find number of components for 95% variance
n_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f'\n💡 {n_95} components are needed to retain ≥ 95% of the total variance.')
print(f'   The first 2 components retain {cumulative_variance[1]:.1%} of variance.')

### 4.4 PCA 2D Projection

In [ ]:
# Project data onto the first 2 principal components
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Distinction': df['Distinction'].map({0: 'No (CGPA < 8.5)', 1: 'Yes (CGPA ≥ 8.5)'})
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Distinction',
                palette={'No (CGPA < 8.5)': '#6366f1', 'Yes (CGPA ≥ 8.5)': '#10b981'},
                s=120, alpha=0.75, edgecolor='white', linewidth=0.5)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('PCA 2D Projection — Student Performance', fontsize=14, fontweight='bold')
plt.legend(title='Distinction', fontsize=10)
plt.tight_layout()
plt.show()

print(f'✅ 2D PCA retains {pca_2d.explained_variance_ratio_.sum():.1%} of the original variance.')

### 4.5 PCA Component Loadings

**Loadings** tell us which original features contribute most to each principal component. This helps interpret what each PC represents.

In [ ]:
# Component loadings (correlation between original features and PCs)
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(len(FEATURES))],
    index=FEATURES
)

print('PCA Component Loadings (first 3 components):')
print(loadings[['PC1', 'PC2', 'PC3']].round(3))
print()

# Visualize loadings for PC1 and PC2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_pc1 = ['#10b981' if v > 0 else '#f43f5e' for v in loadings['PC1']]
axes[0].barh(FEATURES, loadings['PC1'], color=colors_pc1, edgecolor='white')
axes[0].set_xlabel('Loading')
axes[0].set_title(f'PC1 Loadings ({explained_variance[0]:.1%} variance)', fontweight='bold')
axes[0].axvline(x=0, color='gray', linewidth=0.8)

colors_pc2 = ['#10b981' if v > 0 else '#f43f5e' for v in loadings['PC2']]
axes[1].barh(FEATURES, loadings['PC2'], color=colors_pc2, edgecolor='white')
axes[1].set_xlabel('Loading')
axes[1].set_title(f'PC2 Loadings ({explained_variance[1]:.1%} variance)', fontweight='bold')
axes[1].axvline(x=0, color='gray', linewidth=0.8)

plt.tight_layout()
plt.show()

print('\n💡 Interpretation:')
print('   - Features with high absolute loadings contribute strongly to that PC.')
print('   - Green bars = positive contribution, Red bars = negative contribution.')
print('   - Similar loadings across features suggest those features are correlated.')

### 4.6 PCA Biplot

A **biplot** overlays the feature loading vectors on the PCA scatter plot, showing both the data points and the directions of the original features in PC space.

In [ ]:
plt.figure(figsize=(10, 8))

# Plot data points
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['Distinction'],
                      cmap='RdYlGn', s=80, alpha=0.6, edgecolor='white', linewidth=0.3)
plt.colorbar(scatter, label='Distinction (CGPA ≥ 8.5)')

# Plot feature vectors (loadings)
loading_scale = 3  # Scale arrows for visibility
for i, feature in enumerate(FEATURES):
    plt.annotate('',
                 xy=(pca_2d.components_[0, i] * loading_scale,
                     pca_2d.components_[1, i] * loading_scale),
                 xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='#f43f5e', lw=2))
    plt.text(pca_2d.components_[0, i] * loading_scale * 1.15,
             pca_2d.components_[1, i] * loading_scale * 1.15,
             feature, fontsize=9, fontweight='bold', color='#f43f5e')

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})', fontsize=12)
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})', fontsize=12)
plt.title('PCA Biplot — Data Points + Feature Vectors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Arrows pointing in similar directions indicate correlated features.')
print('   Arrows pointing in opposite directions indicate negatively correlated features.')
print('   Arrow length indicates how much that feature contributes to the PCs.')

---

## 5. t-SNE (t-distributed Stochastic Neighbor Embedding)

### 5.1 t-SNE Intuition

t-SNE is a **non-linear** dimensionality reduction technique designed for **visualization**:

1. **Computes pairwise similarities** between data points in high-dimensional space using Gaussian distributions
2. **Maps to low dimensions** (usually 2D) and computes similarities using a Student t-distribution (heavier tails)
3. **Minimizes the KL divergence** between the two similarity distributions
4. Points that are close in high-D tend to stay close in 2D → **preserves local structure**

**Key differences from PCA:**

| Aspect | PCA | t-SNE |
|--------|-----|-------|
| Type | Linear | Non-linear |
| Preserves | Global variance | Local neighborhoods |
| Deterministic | Yes | No (depends on random seed) |
| Scalability | Fast | Slower (O(n²)) |
| Use case | Feature reduction + visualization | Visualization only |
| Interpretable axes | Yes (loadings) | No |

### 5.2 Basic t-SNE Visualization

In [ ]:
# Apply t-SNE
# perplexity must be < n_samples; relates to the number of nearest neighbors considered
perplexity_val = min(30, len(X_scaled) - 1)

tsne = TSNE(
    n_components=2,
    perplexity=perplexity_val,
    init='pca',           # Initialize with PCA for reproducibility
    learning_rate='auto',
    max_iter=1000,
    random_state=42
)
X_tsne = tsne.fit_transform(X_scaled)

tsne_df = pd.DataFrame({
    'tSNE-1': X_tsne[:, 0],
    'tSNE-2': X_tsne[:, 1],
    'Distinction': df['Distinction'].map({0: 'No (CGPA < 8.5)', 1: 'Yes (CGPA ≥ 8.5)'})
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=tsne_df, x='tSNE-1', y='tSNE-2', hue='Distinction',
                palette={'No (CGPA < 8.5)': '#8b5cf6', 'Yes (CGPA ≥ 8.5)': '#06b6d4'},
                s=120, alpha=0.75, edgecolor='white', linewidth=0.5)
plt.title(f't-SNE 2D Visualization (perplexity={perplexity_val})', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(title='Distinction', fontsize=10)
plt.tight_layout()
plt.show()

print(f'✅ t-SNE applied with perplexity={perplexity_val}')
print(f'   KL divergence: {tsne.kl_divergence_:.4f}')

### 5.3 Effect of Perplexity on t-SNE

**Perplexity** is the most important hyperparameter in t-SNE. It can be interpreted as a smooth measure of the effective number of neighbors. Typical values are between 5 and 50.

- **Low perplexity** → focuses on very local structure, may produce many small clusters
- **High perplexity** → considers more neighbors, produces more globular clusters

In [ ]:
max_perp = len(X_scaled) - 1
perplexities = [p for p in [5, 10, 20, 30, 50] if p < max_perp]
if len(perplexities) < 2:
    perplexities = [min(5, max_perp), min(10, max_perp)]

fig, axes = plt.subplots(1, len(perplexities), figsize=(5 * len(perplexities), 5))
if len(perplexities) == 1:
    axes = [axes]

for ax, perp in zip(axes, perplexities):
    tsne_temp = TSNE(n_components=2, perplexity=perp, init='pca',
                     learning_rate='auto', max_iter=1000, random_state=42)
    X_temp = tsne_temp.fit_transform(X_scaled)
    
    scatter = ax.scatter(X_temp[:, 0], X_temp[:, 1], c=df['Distinction'],
                         cmap='viridis', s=60, alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_title(f'Perplexity = {perp}', fontweight='bold', fontsize=12)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

plt.suptitle('t-SNE: Effect of Perplexity', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 Observations:')
print('   - Low perplexity focuses on very local structure (tight, small clusters)')
print('   - High perplexity considers more global context (broader groupings)')
print('   - The "right" perplexity depends on the dataset and what you want to see')

---

## 6. PCA vs t-SNE — Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA
for label, color, marker in [(0, '#6366f1', 'o'), (1, '#10b981', 's')]:
    mask = df['Distinction'] == label
    label_text = 'No Distinction' if label == 0 else 'Distinction'
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=label_text,
                    s=80, alpha=0.7, edgecolor='white', linewidth=0.3, marker=marker)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[0].set_title('PCA (Linear, Global Structure)', fontsize=13, fontweight='bold')
axes[0].legend()

# t-SNE
for label, color, marker in [(0, '#8b5cf6', 'o'), (1, '#06b6d4', 's')]:
    mask = df['Distinction'] == label
    label_text = 'No Distinction' if label == 0 else 'Distinction'
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=color, label=label_text,
                    s=80, alpha=0.7, edgecolor='white', linewidth=0.3, marker=marker)
axes[1].set_xlabel('t-SNE Dimension 1', fontsize=11)
axes[1].set_ylabel('t-SNE Dimension 2', fontsize=11)
axes[1].set_title(f't-SNE (Non-linear, Local Structure)', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('PCA vs t-SNE — Dimensionality Reduction Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 7. PCA for Dimensionality Reduction (Information Retention)

Beyond visualization, PCA is commonly used to **reduce the number of features** before feeding data into a model. Let's see how much information is retained at different numbers of components.

In [ ]:
print('═' * 55)
print('   PCA INFORMATION RETENTION SUMMARY')
print('═' * 55)
print(f'{"Components":<12} {"Variance Retained":>18} {"Reduction":>12}')
print('-' * 45)
n_features = len(FEATURES)
for n in range(1, n_features + 1):
    retained = cumulative_variance[n - 1]
    reduction = (1 - n / n_features) * 100
    star = ' ★' if n == n_95 else ''
    print(f'  {n:<10} {retained:>17.1%} {reduction:>11.0f}%{star}')
print('═' * 55)
print(f'\n★ = Minimum components needed for ≥ 95% variance retention')
print(f'\n💡 Using {n_95} components instead of {n_features} reduces dimensionality')
print(f'   by {(1 - n_95/n_features)*100:.0f}% while retaining {cumulative_variance[n_95-1]:.1%} of information.')

---

## 8. Reconstruction Error

When we reduce dimensions with PCA, some information is lost. The **reconstruction error** measures how well we can reconstruct the original data from the reduced representation.

In [ ]:
errors = []
for n in range(1, n_features + 1):
    pca_n = PCA(n_components=n, random_state=42)
    X_reduced = pca_n.fit_transform(X_scaled)
    X_reconstructed = pca_n.inverse_transform(X_reduced)
    mse = np.mean((X_scaled - X_reconstructed) ** 2)
    errors.append(mse)

plt.figure(figsize=(8, 5))
plt.plot(range(1, n_features + 1), errors, 'o-', color='#f43f5e', linewidth=2, markersize=8)
plt.fill_between(range(1, n_features + 1), errors, alpha=0.1, color='#f43f5e')
plt.xlabel('Number of Components', fontsize=12)
plt.ylabel('Mean Squared Reconstruction Error', fontsize=12)
plt.title('PCA Reconstruction Error', fontsize=14, fontweight='bold')
plt.xticks(range(1, n_features + 1))
plt.tight_layout()
plt.show()

print(f'\n💡 With all {n_features} components: MSE = {errors[-1]:.6f} (perfect reconstruction)')
print(f'   With 2 components: MSE = {errors[1]:.4f}')
print(f'   With {n_95} components (95% var): MSE = {errors[n_95-1]:.4f}')

---

## 9. t-SNE Caveats & Best Practices

t-SNE is powerful but has important limitations that must be understood:

### ⚠️ Common Pitfalls

1. **Cluster sizes are meaningless** — t-SNE can expand dense clusters and compress sparse ones
2. **Distances between clusters are meaningless** — only local distances are preserved
3. **Results vary with random seed** — always set `random_state` for reproducibility
4. **Perplexity matters** — different values can produce very different visualizations
5. **Not for feature engineering** — t-SNE is for visualization only, not for creating features for downstream models

### ✅ Best Practices

1. Always **standardize** data before applying t-SNE
2. Try **multiple perplexity values** and compare results
3. Use `init='pca'` for more **stable and reproducible** results
4. For large datasets, consider using **PCA first** to reduce to ~50 dimensions, then apply t-SNE
5. Don't draw conclusions from cluster sizes or inter-cluster distances

---

## 10. Summary & Key Takeaways

| Aspect | PCA | t-SNE |
|--------|-----|-------|
| **Type** | Linear | Non-linear |
| **Preserves** | Global variance structure | Local neighborhood structure |
| **Deterministic** | ✅ Yes | ❌ No (seed-dependent) |
| **Interpretable axes** | ✅ Yes (loadings) | ❌ No |
| **Speed** | ⚡ Fast (O(np²)) | 🐢 Slower (O(n²)) |
| **Use for features** | ✅ Yes | ❌ No |
| **Use for visualization** | ✅ Yes | ✅ Yes (primary use) |
| **Key parameter** | n_components | perplexity |

### Key Learnings:
1. **PCA** is the go-to linear method for both dimensionality reduction and visualization. Its explained variance metric tells us exactly how much information is retained.
2. **t-SNE** excels at revealing local clusters and patterns invisible to PCA, but should only be used for exploration, not feature engineering.
3. **Standardization** is critical for both methods to prevent features with larger scales from dominating.
4. In practice, PCA and t-SNE are often used **together** — PCA first for dimensionality reduction, then t-SNE for final 2D visualization.

In [ ]:
print('═' * 50)
print('  ✅ WEEK 11–12 NOTEBOOK COMPLETE')
print('═' * 50)
print('  Topics Covered:')
print('  ├── PCA: Explained variance, scree plot,')
print('  │       2D projection, loadings, biplot,')
print('  │       reconstruction error')
print('  ├── t-SNE: 2D visualization, perplexity')
print('  │         effect, caveats & best practices')
print('  └── PCA vs t-SNE comparison')
print('═' * 50)